# SEC 10-K risk factors

The supplied CSV contains Item 1A (Risk Factors) from 15 filings: Adobe, Microsoft, and Oracle, filing years 2021–2025. Each row includes the text, company, filing date, fiscal period end, and SEC source URL.

Run the next cell to read the CSV. The remaining cells download the filings again or collect a different sample. You can skip them if you only need the supplied data.

## Read the supplied CSV
Place the CSV beside this notebook, or use the copy in the course data folder.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

paths = [Path("sec_risk_factors.csv")]
for folder in (Path.cwd(), *Path.cwd().parents):
    paths.append(folder / "data" / "sec_edgar_starter" / "sec_risk_factors.csv")
path = next((p for p in paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Place sec_risk_factors.csv beside the notebook, then rerun this cell.")
risk = pd.read_csv(path, dtype={"cik": str, "sic": str}, keep_default_na=False)
display(risk[["ticker", "filing_date", "report_date", "word_count"]])
print(f"Loaded {len(risk)} filings. Text is in the risk_text column.")

,ticker,filing_date,report_date,word_count
0,ADBE,2021-01-15,2020-11-27,11685
1,ADBE,2022-01-21,2021-12-03,11957
2,ADBE,2023-01-17,2022-12-02,10951
3,ADBE,2024-01-17,2023-12-01,9722
4,ADBE,2025-01-13,2024-11-29,9801
5,MSFT,2021-07-29,2021-06-30,10143
6,MSFT,2022-07-28,2022-06-30,10377
7,MSFT,2023-07-27,2023-06-30,10104
8,MSFT,2024-07-30,2024-06-30,10874
9,MSFT,2025-07-30,2025-06-30,10119


Loaded 15 filings. Text is in the risk_text column.


## Download setup

The course environment has the required packages. In another environment, install them with `%pip install requests pandas beautifulsoup4` if needed.

Enter your name and email for SEC's request header. Change `COMPANIES`, `START_YEAR`, and `END_YEAR` to collect a different sample. The example companies have current SIC 7372 (Prepackaged Software).

Requests are cached and spaced 0.25 seconds apart. A 403 or 429 response stops the download. Retry later.

[SEC API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) · [SEC access guidance](https://www.sec.gov/search-filings/edgar-search-assistance/accessing-edgar-data)

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

CONTACT = os.environ.get("SEC_CONTACT", "").strip()
if not CONTACT:
    CONTACT = input("Your name and contact email (e.g. Jane Doe jane@berkeley.edu): ").strip()
if "@" not in CONTACT or len(CONTACT.split()) < 2:
    raise ValueError("Enter your name and a contact email before downloading.")

# Change these only after deciding on your sample.
COMPANIES = {"MSFT": "0000789019", "ORCL": "0001341439", "ADBE": "0000796343"}
START_YEAR, END_YEAR = 2021, 2025
assert 1995 <= START_YEAR <= END_YEAR <= datetime.now().year

OUT = Path("sec_edgar_output")
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": f"COMPSS211A research {CONTACT}", "Accept-Encoding": "gzip, deflate"}
session = requests.Session()
session.headers.update(HEADERS)
last_request = 0.0

class SECAccessError(RuntimeError):
    pass

def get_text(url):
    """Cache exact response bytes; stop on access blocks; retry transient failures twice."""
    global last_request
    path = CACHE / (hashlib.sha256(url.encode()).hexdigest() + ".bin")
    if path.exists():
        return path.read_bytes().decode("utf-8", errors="replace")
    for attempt in range(3):
        time.sleep(max(0, 0.25 - (time.monotonic() - last_request)))
        last_request = time.monotonic()
        try:
            response = session.get(url, timeout=60)
            if response.status_code in (403, 429):
                raise SECAccessError(f"SEC returned {response.status_code}. Stop and retry later. URL: {url}")
            response.raise_for_status()
        except (requests.ConnectionError, requests.Timeout):
            if attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
            continue
        except requests.HTTPError:
            if response.status_code < 500 or attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
            continue
        path.write_bytes(response.content)
        return response.content.decode("utf-8", errors="replace")

print(f"Sample: {list(COMPANIES)}, filing years {START_YEAR}–{END_YEAR}")

/Users/xutianqin/Desktop/计算机211A/sec_edgar_student_starter/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ValueError: Enter your name and a contact email before downloading.

## List filings

Read the company submissions API, including older records where needed. Keep original 10-K filings within the selected filing-date window; exclude amendments.

`filing_date` is the publication date. `report_date` is the fiscal period end. SIC describes the company's current classification.

In [ ]:
def filings_for_company(ticker, cik):
    payload = json.loads(get_text(f"https://data.sec.gov/submissions/CIK{cik}.json"))
    tables = [pd.DataFrame(payload["filings"]["recent"])]
    low, high = f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"
    for history in payload["filings"].get("files", []):
        if history["filingTo"] >= low and history["filingFrom"] <= high:
            tables.append(pd.DataFrame(json.loads(get_text(
                "https://data.sec.gov/submissions/" + history["name"]
            ))))
    frame = pd.concat(tables, ignore_index=True).drop_duplicates("accessionNumber")
    frame = frame.loc[
        frame["form"].eq("10-K") & frame["filingDate"].between(low, high)
    ].copy()
    frame = frame.rename(columns={"accessionNumber": "accession", "filingDate": "filing_date",
                                  "reportDate": "report_date", "primaryDocument": "primary_document"})
    frame["ticker"], frame["cik"] = ticker, cik
    frame["company"] = payload["name"]
    frame["sic"] = str(payload.get("sic", ""))
    frame["sic_description"] = payload.get("sicDescription", "")
    frame["filing_year"] = pd.to_datetime(frame["filing_date"]).dt.year
    frame["source_url"] = frame.apply(lambda row:
        f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/"
        f"{row['accession'].replace('-', '')}/{row['primary_document']}", axis=1)
    columns = ["ticker", "cik", "company", "sic", "sic_description", "form", "accession",
               "filing_date", "report_date", "filing_year", "primary_document", "source_url"]
    return frame[columns]

filings = pd.concat([filings_for_company(ticker, cik) for ticker, cik in COMPANIES.items()],
                    ignore_index=True).sort_values(["ticker", "filing_date"]).reset_index(drop=True)
if filings.empty:
    raise RuntimeError("No 10-K filings found. Check your companies and date window.")
display(filings[["ticker", "company", "sic", "filing_date", "report_date", "source_url"]])
# Missing company-years remain visible rather than disappearing from the coverage table.
coverage = pd.crosstab(filings.ticker, filings.filing_year).reindex(
    index=list(COMPANIES), columns=range(START_YEAR, END_YEAR + 1), fill_value=0)
display(coverage)
print(f"Found {len(filings)} filings. Review this list before continuing.")

## Extract risk factors

Find the Item 1A heading and the next Item 1B, 1C, or 2 heading. Ignore short contents-table entries and keep the longest candidate.

Formatting varies across filings. Check the extracted text against its source before analysis. Page headers and footers may remain. Failed extractions are recorded separately.

In [ ]:
START = re.compile(r"\bitem\s*1\s*a\s*[.\-:–—]?\s*r\s*i\s*s\s*k\s+f\s*a\s*c\s*t\s*o\s*r\s*s\b", re.I)
END = re.compile(r"\bitem\s*(?:1\s*b\s*[.\-:–—]?\s*unresolve\s*d\s+staff|1\s*c\s*[.\-:–—]?\s*cybersecurity|2\s*[.\-:–—]?\s*properties)\b", re.I)

def extract_risk_factors(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in list(soup.find_all(["script", "style"])):
        tag.decompose()
    for tag in list(soup.find_all(True)):
        if not tag.name:
            continue
        style = re.sub(r"\s+", "", tag.get("style", "").lower())
        if tag.name.lower() == "ix:hidden" or tag.has_attr("hidden") or "display:none" in style:
            tag.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True)).strip()
    starts = list(START.finditer(text))
    boundaries = list(END.finditer(text))
    candidates = []
    for i, start in enumerate(starts):
        # Never join a contents-table start across another Item 1A heading.
        next_start = starts[i + 1].start() if i + 1 < len(starts) else len(text)
        end = next((b for b in boundaries if start.end() < b.start() < next_start), None)
        if end:
            section = text[start.start():end.start()].strip()
            if len(section.split()) >= 250:
                candidates.append((section, start.start(), end.start()))
    if not candidates:
        return {"risk_text": "", "word_count": 0, "extraction_status": "not_extracted",
                "review_status": "not_extracted", "candidate_count": 0,
                "start_offset": None, "end_offset": None}
    section, begin, finish = max(candidates, key=lambda c: len(c[0]))
    return {"risk_text": section, "word_count": len(section.split()),
            "extraction_status": "candidate" if len(candidates) == 1 else "ambiguous_candidate",
            "review_status": "needs_manual_review", "candidate_count": len(candidates),
            "start_offset": begin, "end_offset": finish}

rows = []
retrieved_at = datetime.now(timezone.utc).isoformat()
for record in filings.to_dict("records"):
    try:
        html = get_text(record["source_url"])
        result = extract_risk_factors(html)
        record.update(result)
        record["source_sha256"] = hashlib.sha256(html.encode("utf-8")).hexdigest()
        record["error"] = ""
    except SECAccessError:
        raise  # Do not keep requesting after SEC blocks access.
    except requests.RequestException as error:
        record.update(risk_text="", word_count=0, extraction_status="download_failed",
                      review_status="not_extracted", candidate_count=0,
                      start_offset=None, end_offset=None, source_sha256="", error=str(error))
    record["processed_at_utc"] = retrieved_at
    rows.append(record)
    print(record["ticker"], record["filing_date"], record["extraction_status"], record["word_count"])

risk = pd.DataFrame(rows)
display(risk[["ticker", "filing_date", "word_count", "extraction_status", "review_status"]])

## Check and save

Print the beginning and end of each section and its source URL. Check the boundaries in the original filing.

Save the text CSV, filing list, review checklist, and collection notes in `sec_edgar_output`. Use the checklist to record your review. In Colab, download the files before closing the runtime.

In [ ]:
review = risk.drop(columns=["risk_text"]).copy()
review["beginning"] = risk.risk_text.str[:350]
review["ending"] = risk.risk_text.str[-350:]
review["review_notes"] = ""
for row in review.itertuples():
    print(f"\n{row.ticker} | {row.filing_date} | {row.source_url}")
    print("BEGIN:", row.beginning)
    print("END:", row.ending)

risk.to_csv(OUT / "sec_risk_factors.csv", index=False)
review.to_csv(OUT / "extraction_review.csv", index=False)
filings.to_csv(OUT / "filing_manifest.csv", index=False)
manifest = {
    "created_at_utc": retrieved_at,
    "source": "SEC company submissions API + primary 10-K documents",
    "companies": COMPANIES, "filing_years": [START_YEAR, END_YEAR],
    "selection": "All original 10-K filings for selected CIKs in the inclusive filing-date window; no keyword filtering.",
    "industry_metadata": "Current submissions SIC, not historical industry membership.",
    "parser": "Visible HTML text; longest >=250-word Item 1A Risk Factors candidate ending at next Item 1B/1C/2; manual review required.",
    "rows": len(risk), "status_counts": risk.extraction_status.value_counts().to_dict(),
    "limitations": ["Convenience sample; no causal or sector-wide inference.",
                     "Risk factors are one section of a 10-K filing.",
                     "filing_date differs from report_date; confirm timing for your question.",
                     "Missing/extraction failures are not evidence of silence."]
}
(OUT / "provenance.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved files in {OUT.resolve()}")